# Fadhma-300M → Tarifit Fine-Tuning on Corpus V1.1

**Goal:** fine-tune `agbalu/Fadhma-300M` on the same Tarifit V1.1 corpus used in the other experiments.

- Train: `mms_corpus_v1_1/train`
- Validation: cleaned 128-example subset from `mms_corpus_v1_1/validation`
- Tokenizer: `mms_tokenizer_v1_1`
- Greedy CTC decoding, no external LM
- CER primary, WER also reported

In [ ]:
# Cell 1 — Mount Google Drive and check GPU

from google.colab import drive
drive.mount("/content/drive")

import torch
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("Device:", device)
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print("GPU memory:", round(torch.cuda.get_device_properties(0).total_memory / 1024**3, 1), "GB")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Device: cuda
GPU: Tesla T4
GPU memory: 14.6 GB


In [ ]:
# Cell 2 — Install required packages

!pip -q install -U transformers accelerate datasets jiwer soundfile tqdm
!pip -q install "pandas==2.2.3"

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.3/57.3 kB 6.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.1/12.1 MB 100.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 559.1/559.1 kB 45.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 80.2/80.2 kB 4.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.1/50.1 MB 17.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 97.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 95.8 MB/s eta 0:00:00


In [ ]:
# Cell 3 — Define the V1.1 experiment paths

from pathlib import Path

PROJECT_ROOT = Path(
    "/content/drive/Othercomputers/My MacBook Pro/tarifit_asr_tfm"
)

TRAIN_DIR = PROJECT_ROOT / "data" / "processed" / "mms_corpus_v1_1" / "train"
VALIDATION_DIR = PROJECT_ROOT / "data" / "processed" / "mms_corpus_v1_1" / "validation"
TARIFIT_VOCAB_PATH = PROJECT_ROOT / "data" / "processed" / "mms_tokenizer_v1_1" / "vocab.json"

OMNI_REFERENCE_DIR = PROJECT_ROOT / "models" / "omniasr_w2v_300m_tarifit_v1"

OUTPUT_DIR = PROJECT_ROOT / "models" / "fadhma_300m_tarifit_v1_1"
BEST_BACKUP_DIR = PROJECT_ROOT / "models" / "fadhma_300m_tarifit_v1_1_best_backup"
FINAL_BEST_DIR = PROJECT_ROOT / "models" / "fadhma_300m_tarifit_v1_1_best"
RESULTS_DIR = PROJECT_ROOT / "results" / "fadhma_300m_tarifit_v1_1"

for path in [OUTPUT_DIR, BEST_BACKUP_DIR, FINAL_BEST_DIR, RESULTS_DIR]:
    path.mkdir(parents=True, exist_ok=True)

FADHMA_ID = "agbalu/Fadhma-300M"
BAD_VAL_INDICES = {111, 118, 126, 127, 130}

print("Train exists:", TRAIN_DIR.exists(), TRAIN_DIR)
print("Validation exists:", VALIDATION_DIR.exists(), VALIDATION_DIR)
print("Tokenizer exists:", TARIFIT_VOCAB_PATH.exists(), TARIFIT_VOCAB_PATH)

Train exists: True /content/drive/Othercomputers/My MacBook Pro/tarifit_asr_tfm/data/processed/mms_corpus_v1_1/train
Validation exists: True /content/drive/Othercomputers/My MacBook Pro/tarifit_asr_tfm/data/processed/mms_corpus_v1_1/validation
Tokenizer exists: True /content/drive/Othercomputers/My MacBook Pro/tarifit_asr_tfm/data/processed/mms_tokenizer_v1_1/vocab.json


In [ ]:
# Cell 4 — Load the V1.1 train set and cleaned validation set

from datasets import load_from_disk

train_ds = load_from_disk(str(TRAIN_DIR))
val_ds = load_from_disk(str(VALIDATION_DIR))

valid_val_indices = [i for i in range(len(val_ds)) if i not in BAD_VAL_INDICES]
val_clean_ds = val_ds.select(valid_val_indices)

print("Training examples:", len(train_ds))
print("Training duration:", round(sum(x["input_length"] for x in train_ds) / 16000 / 3600, 2), "hours")
print("Original validation examples:", len(val_ds))
print("Clean validation examples:", len(val_clean_ds))

assert len(train_ds) == 1472
assert len(val_clean_ds) == 128

Training examples: 1472
Training duration: 4.64 hours
Original validation examples: 133
Clean validation examples: 128


In [ ]:
# Cell 5 — Load the Tarifit tokenizer

from transformers import Wav2Vec2CTCTokenizer

tokenizer = Wav2Vec2CTCTokenizer(
    vocab_file=str(TARIFIT_VOCAB_PATH),
    unk_token="[UNK]",
    pad_token="[PAD]",
    word_delimiter_token="|"
)

print("Tarifit vocabulary size:", len(tokenizer))
print("PAD / CTC blank ID:", tokenizer.pad_token_id)
print("Sample reference:", tokenizer.decode(train_ds[0]["labels"], group_tokens=False))

Tarifit vocabulary size: 38
PAD / CTC blank ID: 35
Sample reference: rexbar asbḥan n yasuɛ lmasiḥ sufuss n matta


In [ ]:
# Cell 6 — Create the Fadhma processor with the Tarifit tokenizer

from transformers import AutoFeatureExtractor, Wav2Vec2Processor

feature_extractor = AutoFeatureExtractor.from_pretrained(FADHMA_ID)

processor = Wav2Vec2Processor(
    feature_extractor=feature_extractor,
    tokenizer=tokenizer
)

print("Sampling rate:", feature_extractor.sampling_rate)
print("Audio normalization:", feature_extractor.do_normalize)
print("Tarifit output vocabulary:", len(tokenizer))

preprocessor_config.json:   0%|          | 0.00/214 [00:00<?, ?B/s]

Sampling rate: 16000
Audio normalization: True
Tarifit output vocabulary: 38


In [ ]:
# Cell 7 — Load Fadhma-300M with a new Tarifit CTC head

from transformers import Wav2Vec2ForCTC

model = Wav2Vec2ForCTC.from_pretrained(
    FADHMA_ID,
    vocab_size=len(tokenizer),
    pad_token_id=tokenizer.pad_token_id,
    ctc_loss_reduction="mean",
    ctc_zero_infinity=True,
    ignore_mismatched_sizes=True,
)

model.freeze_feature_encoder()
model.gradient_checkpointing_enable()
model.config.mask_time_prob = 0.05
model.config.layerdrop = 0.0

model = model.to(device)

print("Starting model:", FADHMA_ID)
print("Total parameters:", f"{model.num_parameters():,}")
print("Trainable parameters:", f"{sum(p.numel() for p in model.parameters() if p.requires_grad):,}")
print("Tarifit output vocabulary:", model.config.vocab_size)
print("CTC head:", model.lm_head)

config.json:   0%|          | 0.00/2.06k [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 1.26GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/424 [00:00<?, ?it/s]

[transformers] Wav2Vec2ForCTC LOAD REPORT from: agbalu/Fadhma-300M
Key            | Status   |                                                                                           
---------------+----------+-------------------------------------------------------------------------------------------
lm_head.bias   | MISMATCH | Reinit due to size mismatch - ckpt: torch.Size([40]) vs model:torch.Size([38])            
lm_head.weight | MISMATCH | Reinit due to size mismatch - ckpt: torch.Size([40, 1024]) vs model:torch.Size([38, 1024])

Notes:
- MISMATCH:	ckpt weights were loaded, but they did not match the original empty weight shapes.


Starting model: agbalu/Fadhma-300M
Total parameters: 315,477,670
Trainable parameters: 311,267,494
Tarifit output vocabulary: 38
CTC head: Linear(in_features=1024, out_features=38, bias=True)


In [ ]:
# Cell 8 — Verify CTC feasibility

import torch

def minimum_ctc_frames(labels):
    repeats = sum(labels[i] == labels[i - 1] for i in range(1, len(labels)))
    return len(labels) + repeats

def check_ctc_feasibility(ds, name):
    invalid = []
    for i, example in enumerate(ds):
        output_frames = int(
            model._get_feat_extract_output_lengths(
                torch.tensor(example["input_length"])
            ).item()
        )
        minimum_frames = minimum_ctc_frames(example["labels"])
        if minimum_frames > output_frames:
            invalid.append((i, output_frames, minimum_frames))
    print(name, "total:", len(ds), "valid:", len(ds) - len(invalid), "invalid:", len(invalid))
    return invalid

bad_train = check_ctc_feasibility(train_ds, "TRAIN")
bad_val = check_ctc_feasibility(val_clean_ds, "CLEAN VALIDATION")

assert len(bad_train) == 0
assert len(bad_val) == 0

TRAIN total: 1472 valid: 1472 invalid: 0
CLEAN VALIDATION total: 128 valid: 128 invalid: 0


In [ ]:
# Cell 9 — Create the CTC data collator

from dataclasses import dataclass
from typing import Union

@dataclass
class DataCollatorCTCWithPadding:
    processor: any
    padding: Union[bool, str] = True

    def __call__(self, features):
        input_features = [{"input_values": x["input_values"]} for x in features]
        label_features = [{"input_ids": x["labels"]} for x in features]

        batch = self.processor.pad(
            input_features,
            padding=self.padding,
            return_tensors="pt"
        )

        labels_batch = self.processor.tokenizer.pad(
            label_features,
            padding=self.padding,
            return_tensors="pt"
        )

        batch["labels"] = labels_batch["input_ids"].masked_fill(
            labels_batch["attention_mask"].ne(1),
            -100
        )

        return batch

data_collator = DataCollatorCTCWithPadding(processor=processor)
print("Data collator ready.")

Data collator ready.


In [ ]:
# Cell 10 — Define normalization and WER/CER metrics

import re
import unicodedata
import numpy as np
from jiwer import wer, cer

def eval_normalize(text):
    text = unicodedata.normalize("NFC", str(text))
    text = text.lower().strip()
    text = re.sub(r"\s+", " ", text)
    return text

def compute_metrics(pred):
    pred_ids = np.argmax(pred.predictions, axis=-1)

    label_ids = pred.label_ids.copy()
    label_ids[label_ids == -100] = tokenizer.pad_token_id

    pred_str = processor.batch_decode(pred_ids)
    label_str = tokenizer.batch_decode(label_ids, group_tokens=False)

    pred_str = [eval_normalize(x) for x in pred_str]
    label_str = [eval_normalize(x) for x in label_str]

    return {
        "wer": wer(label_str, pred_str),
        "cer": cer(label_str, pred_str),
    }

print("Metrics ready.")

Metrics ready.


In [ ]:
# Cell 11 — Recover the previous OmniASR optimization settings

import torch

args_files = sorted(OMNI_REFERENCE_DIR.glob("checkpoint-*/training_args.bin"))

if not args_files:
    raise FileNotFoundError(
        "No training_args.bin found in the OmniASR reference experiment."
    )

REFERENCE_ARGS_FILE = args_files[-1]

omni_args = torch.load(
    REFERENCE_ARGS_FILE,
    map_location="cpu",
    weights_only=False
)

for field in [
    "learning_rate",
    "per_device_train_batch_size",
    "per_device_eval_batch_size",
    "gradient_accumulation_steps",
    "num_train_epochs",
    "warmup_steps",
    "weight_decay",
    "seed",
]:
    print(field, "=", getattr(omni_args, field, None))

learning_rate = 0.0001
per_device_train_batch_size = 1
per_device_eval_batch_size = 1
gradient_accumulation_steps = 8
num_train_epochs = 8
warmup_steps = 100
weight_decay = 0.01
seed = 42


In [ ]:
# Cell 12 — Run a forward/backward smoke test

import torch

longest_idx = max(range(len(train_ds)), key=lambda i: train_ds[i]["input_length"])
example = train_ds[longest_idx]

batch = data_collator([example])
batch = {k: v.to(device) for k, v in batch.items()}

model.train()
model.zero_grad(set_to_none=True)

if torch.cuda.is_available():
    torch.cuda.empty_cache()
    torch.cuda.reset_peak_memory_stats()

with torch.autocast(
    device_type="cuda" if torch.cuda.is_available() else "cpu",
    dtype=torch.float16 if torch.cuda.is_available() else torch.bfloat16,
    enabled=torch.cuda.is_available()
):
    outputs = model(**batch)
    smoke_loss = outputs.loss

print("Longest duration:", round(example["input_length"] / 16000, 2), "seconds")
print("Smoke-test loss:", float(smoke_loss))

smoke_loss.backward()

if torch.cuda.is_available():
    print("Peak GPU memory:", round(torch.cuda.max_memory_allocated() / 1024**3, 2), "GB")

print("Forward/backward successful.")
model.zero_grad(set_to_none=True)

Longest duration: 19.98 seconds
Smoke-test loss: 13.446991920471191


/tmp/ipykernel_895/385171634.py:27: UserWarning: Converting a tensor with requires_grad=True to a scalar may lead to unexpected behavior.
Consider using tensor.detach() first. (Triggered internally at /pytorch/torch/csrc/autograd/generated/python_variable_methods.cpp:836.)
  print("Smoke-test loss:", float(smoke_loss))


Peak GPU memory: 2.4 GB
Forward/backward successful.


In [ ]:
# Cell 13 — Create the controlled Fadhma fine-tuning configuration

from transformers import TrainingArguments

training_args = TrainingArguments(
    output_dir=str(OUTPUT_DIR),

    per_device_train_batch_size=int(getattr(omni_args, "per_device_train_batch_size", 1)),
    per_device_eval_batch_size=int(getattr(omni_args, "per_device_eval_batch_size", 1)),
    gradient_accumulation_steps=int(getattr(omni_args, "gradient_accumulation_steps", 8)),

    learning_rate=float(getattr(omni_args, "learning_rate", 1e-4)),
    weight_decay=float(getattr(omni_args, "weight_decay", 0.01)),
    warmup_steps=int(getattr(omni_args, "warmup_steps", 100)),
    num_train_epochs=float(getattr(omni_args, "num_train_epochs", 8)),

    eval_strategy="epoch",
    save_strategy="epoch",
    logging_steps=25,

    load_best_model_at_end=True,
    metric_for_best_model="cer",
    greater_is_better=False,

    fp16=torch.cuda.is_available(),
    gradient_checkpointing=True,
    save_total_limit=2,

    train_sampling_strategy="group_by_length",
    length_column_name="input_length",

    remove_unused_columns=False,
    report_to="none",
    seed=int(getattr(omni_args, "seed", 42)),
)

print("Learning rate:", training_args.learning_rate)
print("Train batch size:", training_args.per_device_train_batch_size)
print("Gradient accumulation:", training_args.gradient_accumulation_steps)
print("Epochs:", training_args.num_train_epochs)
print("Warmup steps:", training_args.warmup_steps)
print("Weight decay:", training_args.weight_decay)
print("Seed:", training_args.seed)

Learning rate: 0.0001
Train batch size: 1
Gradient accumulation: 8
Epochs: 8.0
Warmup steps: 100
Weight decay: 0.01
Seed: 42


In [ ]:
# Cell 14 — Back up the best CER model to Drive

from transformers import TrainerCallback
from pathlib import Path
import math

class BestCERBackupCallback(TrainerCallback):
    def __init__(self, save_dir, processor):
        self.save_dir = Path(save_dir)
        self.processor = processor
        self.best_cer = math.inf

    def on_evaluate(self, args, state, control, metrics=None, model=None, **kwargs):
        if metrics is None or model is None:
            return control

        current_cer = metrics.get("eval_cer")
        if current_cer is None:
            return control

        if current_cer < self.best_cer:
            self.best_cer = current_cer

            model.save_pretrained(
                self.save_dir,
                safe_serialization=True
            )
            self.processor.save_pretrained(self.save_dir)

            with open(self.save_dir / "best_cer.txt", "w", encoding="utf-8") as f:
                f.write(
                    f"best_cer={self.best_cer:.10f}\n"
                    f"epoch={state.epoch}\n"
                    f"global_step={state.global_step}\n"
                )

            print(f"\n✓ Saved best-CER backup: CER={self.best_cer:.6f}, epoch={state.epoch}")

        return control

best_backup_callback = BestCERBackupCallback(
    BEST_BACKUP_DIR,
    processor
)

print("Backup directory:", BEST_BACKUP_DIR)

Backup directory: /content/drive/Othercomputers/My MacBook Pro/tarifit_asr_tfm/models/fadhma_300m_tarifit_v1_1_best_backup


In [ ]:
# Cell 15 — Create the Trainer

from transformers import Trainer

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=val_clean_ds,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
    callbacks=[best_backup_callback],
)

print("Trainer ready.")
print("Train examples:", len(train_ds))
print("Validation examples:", len(val_clean_ds))

Trainer ready.
Train examples: 1472
Validation examples: 128


In [ ]:
# Cell 16 — Start Fadhma-300M to Tarifit V1.1 fine-tuning

train_result = trainer.train()

Epoch,Training Loss,Validation Loss,Wer,Cer
1,6.322423,3.587450,0.982471,0.523060
2,5.397253,3.946068,0.990401,0.533640
3,3.687932,3.493150,0.960351,0.520879
4,3.448377,3.444050,0.975793,0.520233


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]


✓ Saved best-CER backup: CER=0.523060, epoch=1.0


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

...


✓ Saved best-CER backup: CER=0.520879, epoch=3.0


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]


✓ Saved best-CER backup: CER=0.520233, epoch=4.0


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

In [ ]:
# Cell 17A — Check whether the trained Fadhma model was saved to Drive

from pathlib import Path

PROJECT_ROOT = Path(
    "/content/drive/Othercomputers/My MacBook Pro/tarifit_asr_tfm"
)

BEST_BACKUP_DIR = (
    PROJECT_ROOT
    / "models"
    / "fadhma_300m_tarifit_v1_1_best_backup"
)

OUTPUT_DIR = (
    PROJECT_ROOT
    / "models"
    / "fadhma_300m_tarifit_v1_1"
)

print("Best backup exists:", BEST_BACKUP_DIR.exists())

if BEST_BACKUP_DIR.exists():
    print("\nBest-backup files:")
    for file in BEST_BACKUP_DIR.iterdir():
        print(" -", file.name)

print("\nTraining checkpoints:")

if OUTPUT_DIR.exists():
    for checkpoint in sorted(OUTPUT_DIR.glob("checkpoint-*")):
        print("\n", checkpoint.name)

        for file in checkpoint.iterdir():
            if (
                "model" in file.name
                or "trainer_state" in file.name
            ):
                print("   ", file.name)

Best backup exists: True

Best-backup files:
 - config.json
 - tokenizer_config.json
 - added_tokens.json
 - vocab.json
 - processor_config.json
 - best_cer.txt

Training checkpoints:

 checkpoint-184
    trainer_state.json

 checkpoint-368
    trainer_state.json

 checkpoint-552
    trainer_state.json


In [ ]:
# Cell 17B — Check the CER and epoch of the saved best model

BEST_INFO = BEST_BACKUP_DIR / "best_cer.txt"

if BEST_INFO.exists():
    print(BEST_INFO.read_text())
else:
    print("best_cer.txt was not found.")

best_cer=0.5202326145
epoch=4.0
global_step=736



In [ ]:
# Cell 17E — Remount Google Drive

from google.colab import drive

drive.mount("/content/drive", force_remount=True)

print("Drive mounted.")

Mounted at /content/drive
Drive mounted.


In [ ]:
# Cell 17 — Save and verify the best model weights

trainer.save_model(str(FINAL_BEST_DIR))
processor.save_pretrained(str(FINAL_BEST_DIR))

print("Best checkpoint:", trainer.state.best_model_checkpoint)
print("Best metric:", trainer.state.best_metric)

weight_files = [
    p.name
    for p in FINAL_BEST_DIR.iterdir()
    if p.name in {
        "model.safetensors",
        "pytorch_model.bin",
        "model.safetensors.index.json",
        "pytorch_model.bin.index.json",
    }
]

print("Saved weight files:", weight_files)
assert weight_files, "ERROR: no model weight file was saved."
print("✓ Model weights verified on Drive.")

NameError: name 'trainer' is not defined

In [ ]:
# Cell 17C — Inspect the Fadhma best-backup folder

from pathlib import Path

PROJECT_ROOT = Path(
    "/content/drive/Othercomputers/My MacBook Pro/tarifit_asr_tfm"
)

BEST_BACKUP_DIR = (
    PROJECT_ROOT
    / "models"
    / "fadhma_300m_tarifit_v1_1_best_backup"
)

print("Backup directory:", BEST_BACKUP_DIR)
print("Exists:", BEST_BACKUP_DIR.exists())

if BEST_BACKUP_DIR.exists():
    print("\nFiles:")
    for path in sorted(BEST_BACKUP_DIR.rglob("*")):
        if path.is_file():
            print(
                path.relative_to(BEST_BACKUP_DIR),
                "—",
                round(path.stat().st_size / 1024**2, 2),
                "MB"
            )

Backup directory: /content/drive/Othercomputers/My MacBook Pro/tarifit_asr_tfm/models/fadhma_300m_tarifit_v1_1_best_backup
Exists: True

Files:
added_tokens.json — 0.0 MB
best_cer.txt — 0.0 MB
config.json — 0.0 MB
processor_config.json — 0.0 MB
tokenizer_config.json — 0.0 MB
vocab.json — 0.0 MB


In [ ]:
# Cell 17D — Search all Fadhma folders for saved model weights

MODELS_DIR = PROJECT_ROOT / "models"

weight_files = []

for path in MODELS_DIR.rglob("*"):
    if not path.is_file():
        continue

    if "fadhma" not in str(path).lower():
        continue

    if (
        path.name == "model.safetensors"
        or path.name == "pytorch_model.bin"
        or path.name.endswith(".safetensors")
    ):
        weight_files.append(path)

print("Fadhma weight files found:", len(weight_files))

for path in weight_files:
    print(
        path,
        "—",
        round(path.stat().st_size / 1024**3, 3),
        "GB"
    )

Fadhma weight files found: 0


In [ ]:
# Cell 18A — Load the saved best Fadhma model and clean validation set on CPU

from pathlib import Path
import torch
from datasets import load_from_disk
from transformers import Wav2Vec2ForCTC, Wav2Vec2Processor

PROJECT_ROOT = Path(
    "/content/drive/Othercomputers/My MacBook Pro/tarifit_asr_tfm"
)

BEST_BACKUP_DIR = (
    PROJECT_ROOT
    / "models"
    / "fadhma_300m_tarifit_v1_1_best_backup"
)

VALIDATION_DIR = (
    PROJECT_ROOT
    / "data"
    / "processed"
    / "mms_corpus_v1_1"
    / "validation"
)

BAD_VAL_INDICES = {111, 118, 126, 127, 130}

device = torch.device("cpu")

# Load the saved epoch-4 model
model = Wav2Vec2ForCTC.from_pretrained(
    BEST_BACKUP_DIR
).to(device)

processor = Wav2Vec2Processor.from_pretrained(
    BEST_BACKUP_DIR
)

tokenizer = processor.tokenizer

model.eval()

# Load and clean validation set
val_ds = load_from_disk(
    str(VALIDATION_DIR)
)

valid_val_indices = [
    i
    for i in range(len(val_ds))
    if i not in BAD_VAL_INDICES
]

val_clean_ds = val_ds.select(
    valid_val_indices
)

print("Model loaded from:", BEST_BACKUP_DIR)
print("Device:", device)
print("Clean validation examples:", len(val_clean_ds))

OSError: Error no file named model.safetensors, or pytorch_model.bin, found in directory /content/drive/Othercomputers/My MacBook Pro/tarifit_asr_tfm/models/fadhma_300m_tarifit_v1_1_best_backup.

In [ ]:
# Cell 18 — Evaluate the best model with greedy CTC decoding

import torch
from tqdm.auto import tqdm
from jiwer import wer, cer

trainer.model.eval()

references = []
predictions = []

for example in tqdm(val_clean_ds, desc="Fadhma -> Tarifit V1.1 validation"):
    input_values = torch.tensor(
        example["input_values"],
        dtype=torch.float32
    ).unsqueeze(0).to(device)

    with torch.inference_mode():
        logits = trainer.model(input_values=input_values).logits

    pred_ids = torch.argmax(logits, dim=-1)
    prediction = processor.batch_decode(pred_ids)[0]
    reference = tokenizer.decode(example["labels"], group_tokens=False)

    predictions.append(eval_normalize(prediction))
    references.append(eval_normalize(reference))

final_wer = wer(references, predictions)
final_cer = cer(references, predictions)

print("=" * 65)
print("FADHMA-300M -> TARIFIT V1.1 — CLEAN VALIDATION")
print("=" * 65)
print("Segments :", len(references))
print(f"WER      : {final_wer * 100:.2f}%")
print(f"CER      : {final_cer * 100:.2f}%")

In [ ]:
# Cell 18B — Evaluate the saved best Fadhma model with greedy CTC decoding

import re
import unicodedata
import torch
from tqdm.auto import tqdm
from jiwer import wer, cer

def eval_normalize(text):
    text = unicodedata.normalize("NFC", str(text))
    text = text.lower().strip()
    text = re.sub(r"\s+", " ", text)
    return text

references = []
predictions = []

model.eval()

for example in tqdm(
    val_clean_ds,
    desc="Fadhma -> Tarifit validation"
):
    input_values = torch.tensor(
        example["input_values"],
        dtype=torch.float32
    ).unsqueeze(0).to(device)

    with torch.inference_mode():
        logits = model(
            input_values=input_values
        ).logits

    pred_ids = torch.argmax(
        logits,
        dim=-1
    )

    prediction = processor.batch_decode(
        pred_ids
    )[0]

    reference = tokenizer.decode(
        example["labels"],
        group_tokens=False
    )

    predictions.append(
        eval_normalize(prediction)
    )

    references.append(
        eval_normalize(reference)
    )

final_wer = wer(
    references,
    predictions
)

final_cer = cer(
    references,
    predictions
)

print("=" * 65)
print("FADHMA-300M -> TARIFIT V1.1 — CLEAN VALIDATION")
print("=" * 65)
print("Segments :", len(references))
print(f"WER      : {final_wer * 100:.2f}%")
print(f"CER      : {final_cer * 100:.2f}%")

In [ ]:
# Cell 19 — Save predictions and experiment summary

import pandas as pd
from jiwer import wer, cer

results_df = pd.DataFrame({
    "clean_validation_index": range(len(val_clean_ds)),
    "original_validation_index": valid_val_indices,
    "duration_seconds": [x["input_length"] / 16000 for x in val_clean_ds],
    "reference": references,
    "prediction": predictions,
    "wer": [wer(r, p) for r, p in zip(references, predictions)],
    "cer": [cer(r, p) for r, p in zip(references, predictions)],
})

predictions_file = RESULTS_DIR / "fadhma_tarifit_v1_1_validation_128.csv"
results_df.to_csv(predictions_file, index=False, encoding="utf-8")

summary_df = pd.DataFrame([{
    "experiment": "Fadhma-300M -> Tarifit V1.1 fine-tuning",
    "initial_model": FADHMA_ID,
    "training_dataset": str(TRAIN_DIR),
    "train_examples": len(train_ds),
    "validation_examples": len(val_clean_ds),
    "decoding": "greedy CTC",
    "language_model": False,
    "wer_percent": final_wer * 100,
    "cer_percent": final_cer * 100,
    "best_checkpoint": trainer.state.best_model_checkpoint,
    "best_metric": trainer.state.best_metric,
}])

summary_file = RESULTS_DIR / "fadhma_tarifit_v1_1_summary.csv"
summary_df.to_csv(summary_file, index=False)

print("Saved predictions:", predictions_file)
print("Saved summary:", summary_file)
display(summary_df)

## Experiment conclusion

Initializing from a Kabyle-adapted OmniASR model appears to provide a modest benefit over direct fine-tuning from the generic OmniASR encoder, suggesting that related-language pre-adaptation may improve character-level transfer to Tarifit.

Fadhma-300M was fine-tuned on the Tarifit V1.1 corpus. The best validation performance was observed at epoch 4, with a WER of 97.58% and a CER of 52.02% on the cleaned 128-segment validation set.